# Optimiser les représentations de texte (text embeddings) pour la recherche d'emploi par IA

### importation des librairies

In [1]:
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import TripletEvaluator

### importation du modèle pré-entrainé

Différents modèles de base sont classés [ici](https://sbert.net/docs/sentence_transformer/training_overview.html#best-base-embedding-models)

In [2]:
model_name = "sentence-transformers/all-distilroberta-v1"
model = SentenceTransformer(model_name)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Télécharger le dataset HF

In [3]:
dataset = load_dataset("Fe2x/ai-job-embedding-finetuning")

README.md:   0%|          | 0.00/580 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/285k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/267k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/757 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/94 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/96 [00:00<?, ? examples/s]

### Évaluer le modèle pré-entraîné sur les données d'évaluation.

In [4]:
evaluator_valid = TripletEvaluator(
    anchors=dataset["validation"]["query"],
    positives=dataset["validation"]["job_description_pos"],
    negatives=dataset["validation"]["job_description_neg"],
    name="ai-job-validation",
)
evaluator_valid(model)

{'ai-job-validation_cosine_accuracy': 0.9468085169792175}

### définir la fonction de perte

In [5]:
loss = MultipleNegativesRankingLoss(model)

### Définir les arguments d'entrainement

In [6]:
num_epochs = 1
batch_size = 16
lr = 2e-5
finetuned_model_name = "distilroberta-ai-job-embeddings"

train_args = SentenceTransformerTrainingArguments(
    output_dir=f"models/{finetuned_model_name}",
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=lr,
    warmup_ratio=0.1,
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss bénéficie de l'absence d'échantillons en double dans un lot.
    eval_strategy="steps",
    eval_steps=100,
    logging_steps=100,
)

### fine-tuner le modèle

In [7]:
%%time
trainer = SentenceTransformerTrainer(
    model=model,
    args=train_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    loss=loss,
    evaluator=evaluator_valid,
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: foulhadj (foulhadj-scc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss


CPU times: user 1min 8s, sys: 14 s, total: 1min 21s
Wall time: 2min 11s


TrainOutput(global_step=48, training_loss=0.7360928058624268, metrics={'train_runtime': 127.395, 'train_samples_per_second': 5.942, 'train_steps_per_second': 0.377, 'total_flos': 0.0, 'train_loss': 0.7360928058624268, 'epoch': 1.0})

### Évaluer le modèle finetuné

In [8]:
evaluator_test = TripletEvaluator(
    anchors=dataset["test"]["query"],
    positives=dataset["test"]["job_description_pos"],
    negatives=dataset["test"]["job_description_neg"],
    name="ai-job-test",
)
print("Validation:", evaluator_valid(model))
print("Test:", evaluator_test(model))

Validation: {'ai-job-validation_cosine_accuracy': 0.9893617033958435}
Test: {'ai-job-test_cosine_accuracy': 1.0}


### Déployer le modèle optimisé vers le hub HF

In [9]:
from huggingface_hub import notebook_login
notebook_login()

In [10]:
model.push_to_hub(f"Fe2x/{finetuned_model_name}")

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmp4193p3x5/model.safetensors    :   0%|          |  551kB /  328MB            

'https://huggingface.co/Fe2x/distilroberta-ai-job-embeddings/commit/1a4a801b7994e8f95d2ed347780f536b92f21436'